# Approval Gate Inside a Skill — Email Drafter

This notebook shows that a skill can own its own HITL gate — not just
collect preferences, but enforce an irreversible-action checkpoint.

The `email-drafter` skill drafts a reply to an email, then calls
`HumanInputTool` with `choices=["send", "discard"]` before the reply
is "sent". The agent invokes the skill and receives the final outcome;
it never sees or controls the approval step.

## What a reader learns

- A skill can be the HITL boundary itself, not just a behaviour wrapper
- This pattern is appropriate for any irreversible action — send, deploy,
  delete, push — where the gate belongs at the action level, not the
  agent or operator level
- Contrasts with the patterns in `ch08.ipynb`:
  - **Example 1** — `HumanInputTool` used by the agent mid-task (agent-initiated)
  - **Example 2** — `request_approval` at the end of the loop (operator-gated,
    result-level)
  - **Here** — gate is skill-encapsulated and action-level; the agent is
    unaware it exists

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama
installed on your local machine and have its LLM hosting service running.
To download Ollama, follow the instructions found on this page:
https://ollama.com/download. After downloading and installing Ollama, you
can start a service by opening a terminal and running `ollama serve`.

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("\u2713 Using Ollama Cloud")

In [ ]:
model = "qwen3.5:397b-cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None

In [ ]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.tools.default import HumanInputTool

enable_console_logging(logging.INFO)

human_input_tool = HumanInputTool()
llm = OllamaLLM(host=host, model=model, think=False, json_prompt_mode=use_cloud)
agent = LLMAgent(llm=llm, tools=[human_input_tool])

In [ ]:
EMAIL = """\
From: sarah.chen@acmecorp.com
To: alex@mycompany.com
Subject: Re: Q3 Partnership Proposal

Hi Alex,

Thanks for sending over the Q3 partnership proposal last week. We reviewed it
with the team and we're excited about the direction.

A few things we'd like to clarify before signing:

1. Revenue share — the proposal mentions 20% for the first year, but our
   standard agreement starts at 15%. Is there flexibility here?
2. The exclusivity clause covers our North America region, but we'd need it
   limited to the retail vertical only.
3. Start date — we're targeting September 1. Does that work on your end?

If these points can be addressed, we'd like to move quickly. Can we get on a
call this week to align?

Best,
Sarah
Sarah Chen | Head of Partnerships | Acme Corp
"""

## Running the Skill

When the agent activates the `email-drafter` skill, it will:

1. Draft a reply addressing Sarah's three points
2. Pause and present the draft via `HumanInputTool` — asking you to **send**
   or **discard**
3. Act on your choice and return the outcome

The agent receives only the final outcome. It has no visibility into the
approval step — that gate lives entirely inside the skill.

In [ ]:
result = await agent.run_with_skill(
    "email-drafter",
    prompt=f"Draft a reply to the following email:\n\n{EMAIL}",
)
print(result.content)